In [1]:
#RGI 7.0 and vas sript
# ============================================================
# 1. IMPORTS
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt

import oggm
import oggm.cfg as cfg
from oggm import utils, workflow, tasks

from oggm.shop import gcm_climate

# Add the old VAS climate_info basename
if "climate_info" not in cfg.BASENAMES:

    cfg.BASENAMES["climate_info"] = (
        "climate_info.json",
        "VAS glacier climate and calibration information."
    )

# JSON compatibility patch for OGGM 1.6.3 + VAS
import json
import numpy as np
from oggm import GlacierDirectory


def write_json_compat(
    self,
    var,
    filename,
    filesuffix=""
):

    def np_convert(obj):

        if isinstance(obj, np.integer):
            return int(obj)

        if isinstance(obj, np.floating):
            return float(obj)

        if isinstance(obj, np.bool_):
            return bool(obj)

        if isinstance(obj, np.ndarray):
            return obj.tolist()

        raise TypeError(
            f"Object of type {type(obj).__name__} "
            "is not JSON serializable"
        )

    fp = self.get_filepath(
        filename,
        filesuffix=filesuffix
    )

    with open(fp, "w") as f:

        json.dump(
            var,
            f,
            default=np_convert
        )


GlacierDirectory.write_json = write_json_compat

# ============================================================
# OGGM 1.6.3 / VAS get_climate_info COMPATIBILITY PATCH
# ============================================================

_original_get_climate_info = GlacierDirectory.get_climate_info


def get_climate_info_compat(
    self,
    input_filesuffix=""
):

    # Get the normal OGGM 1.6.3 climate information
    out = _original_get_climate_info(
        self,
        input_filesuffix=input_filesuffix
    )

    # Merge the older VAS climate_info.json file
    try:

        vas_info = self.read_json(
            "climate_info"
        )

        if isinstance(vas_info, dict):
            out.update(vas_info)

    except FileNotFoundError:

        pass

    return out


GlacierDirectory.get_climate_info = (
    get_climate_info_compat
)

import oggm_vas as vascaling

In [2]:
# ============================================================
# 2. GENERAL CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# RGI 7.0
# ------------------------------------------------------------

RGI_VERSION = "70G"

# Select the RGI first-order region here.
# 1-19 are available.
RGI_REGION = 11


RGI_REGION_NAMES = {
    1: "Alaska",
    2: "Western Canada and USA",
    3: "Arctic Canada North",
    4: "Arctic Canada South",
    5: "Greenland Periphery",
    6: "Iceland",
    7: "Svalbard",
    8: "Scandinavia",
    9: "Russian Arctic",
    10: "North Asia",
    11: "Central Europe",
    12: "Caucasus and Middle East",
    13: "Central Asia",
    14: "South Asia West",
    15: "South Asia East",
    16: "Low Latitudes",
    17: "Southern Andes",
    18: "New Zealand",
    19: "Antarctic and Subantarctic",
}


if RGI_REGION not in RGI_REGION_NAMES:

    raise ValueError(
        "RGI_REGION must be between 1 and 19."
    )


REGION_NAME = RGI_REGION_NAMES[RGI_REGION]


# ------------------------------------------------------------
# Historical reconstruction
# ------------------------------------------------------------

MODEL_START_YEAR = 1901
HISTORICAL_END_YEAR = 2019

BASELINE_CLIMATE = "CRU"


# ------------------------------------------------------------
# Future projection
# ------------------------------------------------------------

RUN_FUTURE = True

GCM = "CCSM4"

PROJECTIONS = [
    "rcp26",
    "rcp45",
    "rcp85"
]

FUTURE_START_YEAR = 2019
FUTURE_END_YEAR = 2100


# ------------------------------------------------------------
# Testing
# ------------------------------------------------------------
#
# Start with 1 glacier.
# Once successful, change to None.
# ------------------------------------------------------------

MAX_GLACIERS = 10



In [3]:
# ============================================================
# 3. WORKING DIRECTORY
# ============================================================

WORKDIR = Path(
    os.path.expanduser(
        f"~/oggm_work/"
        f"vas_rgi70_region{RGI_REGION}_"
        f"cru_{MODEL_START_YEAR}_{HISTORICAL_END_YEAR}"
    )
)

WORKDIR.mkdir(
    parents=True,
    exist_ok=True
)



In [4]:
# ============================================================
# 4. INITIALIZE OGGM-VAS
# ============================================================

print("\nInitializing OGGM-VAS...")

vascaling.initialize()

cfg.PATHS["working_dir"] = str(WORKDIR)

# RGI7
cfg.PARAMS["rgi_version"] = RGI_VERSION

# Climate
cfg.PARAMS["baseline_climate"] = BASELINE_CLIMATE

# Calendar years
cfg.PARAMS["hydro_month_nh"] = 1
cfg.PARAMS["hydro_month_sh"] = 1

# Do not use OGGM's standard multiprocessing initially.
cfg.PARAMS["use_multiprocessing"] = False

cfg.PARAMS["use_intersects"] = False

# Continue when an individual glacier fails.
cfg.PARAMS["continue_on_error"] = True


cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["mp_processes"] = 4


2026-10-06 11:14:41: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-10-06 11:14:41: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-10-06 11:14:41: oggm.cfg: Multiprocessing: using all available processors (N=12)
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `prcp_default_gradient`:`0.0004` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `vas_c_area_m2`:`0.1912` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `vas_gamma_area`:`1.375` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `vas_c_length_m`:`4.5214` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `vas_q_length`:`2.2` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `vas_c_icecap_area_m2`:`1.7013` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `vas_gamma_i


Initializing OGGM-VAS...


In [5]:
# ============================================================
# 5. ADD VAS PARAMETERS
# ============================================================

cfg.PARAMS["prcp_scaling_factor"] = 3.0

cfg.PARAMS["temp_melt"] = 0.0

cfg.PARAMS["temp_all_solid"] = 4.0

cfg.PARAMS["temp_all_liq"] = 2.0

cfg.PARAMS["climate_qc_months"] = 3

cfg.PARAMS["temp_default_gradient"] = -0.0065

cfg.PARAMS["temp_local_gradient_bounds"] = [
    -0.009,
    -0.003
]

cfg.PARAMS["temp_use_local_gradient"] = False

cfg.PARAMS["prcp_default_gradient"] = 0.0004

cfg.PARAMS["mu_star_halfperiod"] = 15

cfg.PARAMS["min_mu_star"] = 5

cfg.PARAMS["max_mu_star"] = 10000

# Use the calibrated VAS bias during model runs
cfg.PARAMS["use_bias_for_run"] = True

2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `prcp_scaling_factor`:`3.0` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: PARAMS['temp_melt'] changed from `-1.0` to `0.0`.
2026-10-06 11:14:41: oggm.cfg: PARAMS['temp_all_solid'] changed from `0.0` to `4.0`.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `climate_qc_months`:`3` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `temp_local_gradient_bounds`:`[-0.009, -0.003]` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `temp_use_local_gradient`:`False` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `mu_star_halfperiod`:`15` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `min_mu_star`:`5` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `max_mu_star`:`10000` to PARAMS.
2026-10-06 11:14:41: oggm.cfg: WARNING: adding an unknown parameter `use_bias_for

In [6]:
# ============================================================
# 6. LOAD RGI 7.0 REGION
# ============================================================

print(
    f"\nLoading RGI 7.0 Region "
    f"{RGI_REGION:02d}: {REGION_NAME}"
)

# OGGM 1.6.3 downloads the official RGI7 global archive
# and extracts the requested regional shapefile.
rgi_file = utils.get_rgi_region_file(
    RGI_REGION,
    version="70G"
)

print(
    "\nRGI file:"
)

print(rgi_file)


rgidf = gpd.read_file(
    rgi_file
)


print(
    "\nNumber of RGI7 glaciers:",
    len(rgidf)
)




Loading RGI 7.0 Region 11: Central Europe

RGI file:
/home/JasperE/OGGM/rgi/RGIV70G/RGI2000-v7.0-G-11_central_europe/RGI2000-v7.0-G-11_central_europe.shp

Number of RGI7 glaciers: 4079


In [7]:
# ============================================================
# 7. NORMALIZE RGI7 ID
# ============================================================

if "rgi_id" in rgidf.columns:

    pass

elif "RGIId" in rgidf.columns:

    rgidf["rgi_id"] = rgidf["RGIId"]

else:

    raise RuntimeError(
        "Could not identify the RGI7 glacier ID column."
    )



In [8]:
# ============================================================
# 8. CHECK GLACIER STATUS
# ============================================================
#
# Unlike RGI6, RGI7 substantially revised the inventory and
# replaced nominal glaciers with real outlines.
#
# We therefore only filter Status if the field exists.
# ============================================================

status_column = None

if "status" in rgidf.columns:

    status_column = "status"

elif "Status" in rgidf.columns:

    status_column = "Status"


if status_column is not None:

    nominal = (
        rgidf[status_column]
        .astype(str)
        .str.strip()
        .isin(["2", "9"])
    )

    print(
        "Potential nominal/unassigned glaciers:",
        nominal.sum()
    )

    if nominal.any():

        rgidf = rgidf.loc[
            ~nominal
        ].copy()

else:

    print(
        "No RGI6-style Status field found; "
        "no nominal-glacier filtering required."
    )


print(
    "Glaciers after inventory filtering:",
    len(rgidf)
)



No RGI6-style Status field found; no nominal-glacier filtering required.
Glaciers after inventory filtering: 4079


In [9]:
# ============================================================
# 9. OPTIONAL TEST MODE
# ============================================================

if MAX_GLACIERS is not None:

    rgidf = rgidf.head(
        MAX_GLACIERS
    ).copy()

    print(
        "\nTEST MODE:"
    )

    print(
        "Using glaciers:",
        len(rgidf)
    )




TEST MODE:
Using glaciers: 10


In [10]:
# ============================================================
# 10. INITIALIZE RGI7 PREPROCESSED GLACIER DIRECTORIES
# ============================================================
#
# RGI-TOPO provides official OGGM glacier directories for
# RGI7 glaciers.
#
# The current OGGM RGI7 Level-1 path is:
#
#   .../rgitopo/2026.1/all_dems/RGI70G/b_010/L1/
#
# Level 1 provides the local glacier topography/grid.
# ============================================================

PREPRO_BASE_URL = (
    "https://cluster.klima.uni-bremen.de/"
    "~oggm/gdirs/oggm_v1.6/"
    "rgitopo/2026.1/all_dems/"
)


print(
    "\nInitializing RGI7 preprocessed glacier directories..."
)


gdirs = workflow.init_glacier_directories(
    rgidf,
    from_prepro_level=1,
    prepro_border=10,
    prepro_rgi_version="70G",
    prepro_base_url=PREPRO_BASE_URL
)


print(
    "\nGlacier directories:",
    len(gdirs)
)



Initializing RGI7 preprocessed glacier directories...


2026-10-06 11:14:41: oggm.workflow: init_glacier_directories from prepro level 1 on 10 glaciers.
2026-10-06 11:14:41: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 10 glaciers



Glacier directories: 10


2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00002) glacier_masks
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00001) glacier_masks
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00004) glacier_masks
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00003) glacier_masks
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00001) process_dem
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00002) process_dem
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00003) process_dem
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00004) process_dem
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00006) glacier_masks
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00005) glacier_masks
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00005) process_dem
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00006) process_dem
2026-10-06 11:14:44: oggm.core.gis: (RGI2000-v7.0-G-11-00008) glacier_masks
2026-10-06 11:14:44: ogg

In [11]:
# ============================================================
# 11. PROCESS GLACIER MASKS
# ============================================================

print(
    "\nProcessing glacier masks..."
)


workflow.execute_entity_task(
    tasks.glacier_masks,
    gdirs
)


print(
    "Glacier masks complete."
)



2026-10-06 11:14:44: oggm.workflow: Execute entity tasks [glacier_masks] on 10 glaciers



Processing glacier masks...
Glacier masks complete.


In [12]:
# ============================================================
# 12. PROCESS CRU CLIMATE
# ============================================================

print(
    f"\nProcessing CRU climate "
    f"{MODEL_START_YEAR}-{HISTORICAL_END_YEAR}..."
)


workflow.execute_entity_task(
    tasks.process_climate_data,
    gdirs,
    y0=MODEL_START_YEAR,
    y1=HISTORICAL_END_YEAR
)


print(
    "Historical climate processing complete."
)



2026-10-06 11:14:44: oggm.workflow: Execute entity tasks [process_climate_data] on 10 glaciers



Processing CRU climate 1901-2019...
Historical climate processing complete.


In [13]:
# ============================================================
# 13. LOAD VAS CRU REFERENCE t* TABLE
# ============================================================
#
# OGGM 1.6.3 does not contain an RGI7 WGMS-reference mapping
# compatible with the old VAS calibration routine.
#
# VAS local_t_star() accepts a custom reference table based
# on longitude, latitude, t_star and bias.
#
# We therefore use the calibrated VAS CRU reference table
# shipped with OGGM-VAS and apply it spatially to the RGI7
# glaciers.
# ============================================================

print(
    "\nLoading VAS CRU reference t* table..."
)


# ------------------------------------------------------------
# Locate the reference table shipped with OGGM-VAS
# ------------------------------------------------------------

ref_table_file = (
    vascaling.get_ref_tstars_filepath(
        "vas_ref_tstars_rgi6_cru4.csv"
    )
)


print(
    "VAS reference table:"
)

print(
    ref_table_file
)


# ------------------------------------------------------------
# Read reference table
# ------------------------------------------------------------

ref_df = pd.read_csv(
    ref_table_file
)


print(
    "\nReference glaciers:",
    len(ref_df)
)


print(
    "\nReference table columns:"
)

print(
    ref_df.columns.tolist()
)


# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------

required_columns = {
    "lon",
    "lat",
    "tstar",
    "bias"
}


missing_columns = (
    required_columns
    - set(ref_df.columns)
)


if missing_columns:

    raise RuntimeError(
        "The VAS reference table is missing "
        f"the following columns: {missing_columns}"
    )


# ------------------------------------------------------------
# Save a copy in the RGI7 experiment directory
# ------------------------------------------------------------

REF_TABLE_PATH = (
    WORKDIR /
    "vas_ref_tstars_rgi7_spatial_cru.csv"
)


ref_df.to_csv(
    REF_TABLE_PATH,
    index=False
)


print(
    "\nReference table copied to:"
)

print(
    REF_TABLE_PATH
)


# ------------------------------------------------------------
# Check the climate/MB parameters
# ------------------------------------------------------------

print(
    "\nVAS reference calibration table loaded."
)

print(
    "The table will be spatially interpolated "
    "to the RGI7 glaciers."
)


Loading VAS CRU reference t* table...
VAS reference table:
/home/JasperE/oggm-vas-rgi7/oggm_vas/data/vas_ref_tstars_rgi6_cru4.csv

Reference glaciers: 268

Reference table columns:
['Unnamed: 0', 'lon', 'lat', 'n_mb_years', 'tstar', 'bias']

Reference table copied to:
/home/JasperE/oggm_work/vas_rgi70_region11_cru_1901_2019/vas_ref_tstars_rgi7_spatial_cru.csv

VAS reference calibration table loaded.
The table will be spatially interpolated to the RGI7 glaciers.


In [14]:
# ============================================================
# 14. VAS LOCAL t* CALIBRATION
# ============================================================

print(
    "\nCalibrating local VAS t* and mu* values..."
)

cfg.PARAMS["run_mb_calibration"] = False

workflow.execute_entity_task(
    vascaling.local_t_star,
    gdirs,
    ref_df=ref_df
)

print(
    "VAS calibration complete."
)

2026-10-06 11:14:49: oggm.cfg: WARNING: adding an unknown parameter `run_mb_calibration`:`False` to PARAMS.
2026-10-06 11:14:49: oggm.workflow: Execute entity tasks [local_t_star] on 10 glaciers



Calibrating local VAS t* and mu* values...


2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00001) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00002) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00003) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00004) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00006) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00005) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00007) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00008) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00009) local_t_star
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00010) local_t_star


VAS calibration complete.


2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00002) run_historic_from_climate_data_hist1901_2019
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00003) run_historic_from_climate_data_hist1901_2019
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00004) run_historic_from_climate_data_hist1901_2019
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00001) run_historic_from_climate_data_hist1901_2019
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00002) find_start_area
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00004) find_start_area
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00003) find_start_area
2026-10-06 11:14:49: oggm_vas.core: (RGI2000-v7.0-G-11-00001) find_start_area


In [15]:
# ============================================================
# 15. CHECK VAS CALIBRATION
# ============================================================

calibration_records = []

valid_gdirs = []

failed_calibration = []


for gdir in gdirs:

    try:

        cal = gdir.read_json(
            "vascaling_mustar"
        )


        record = {
            "rgi_id":
                gdir.rgi_id,

            "t_star":
                cal.get(
                    "t_star",
                    np.nan
                ),

            "bias":
                cal.get(
                    "bias",
                    np.nan
                ),

            "mu_star":
                cal.get(
                    "mu_star",
                    np.nan
                )
        }


        calibration_records.append(
            record
        )


        if (
            np.isfinite(
                record["t_star"]
            )
            and
            np.isfinite(
                record["bias"]
            )
            and
            np.isfinite(
                record["mu_star"]
            )
        ):

            valid_gdirs.append(
                gdir
            )

        else:

            failed_calibration.append(
                gdir.rgi_id
            )


    except Exception:

        failed_calibration.append(
            gdir.rgi_id
        )


print(
    "\nVAS calibration:"
)


print(
    "  Valid:",
    len(valid_gdirs)
)


print(
    "  Failed:",
    len(failed_calibration)
)


pd.DataFrame(
    calibration_records
).to_csv(
    WORKDIR /
    "vas_calibration_rgi70.csv",
    index=False
)


pd.DataFrame(
    {
        "rgi_id":
            failed_calibration
    }
).to_csv(
    WORKDIR /
    "failed_vas_calibration.csv",
    index=False
)


gdirs = valid_gdirs


if len(gdirs) == 0:

    raise RuntimeError(
        "No glaciers passed VAS calibration."
    )




VAS calibration:
  Valid: 10
  Failed: 0


In [16]:
# ============================================================
# CREATE / REBUILD gridded_data.nc
# ============================================================

test_gdir = gdirs[0]

print("Glacier:", test_gdir.rgi_id)

print("\nRunning glacier_masks with reset=True...")

tasks.glacier_masks(
    test_gdir,
    reset=True
)

gridded_file = test_gdir.get_filepath(
    "gridded_data"
)

print("\nGridded data file:")
print(gridded_file)

print(
    "Exists:",
    os.path.exists(gridded_file)
)

2026-10-06 11:14:49: oggm.core.gis: (RGI2000-v7.0-G-11-00001) glacier_masks


Glacier: RGI2000-v7.0-G-11-00001

Running glacier_masks with reset=True...

Gridded data file:
/home/JasperE/oggm_work/vas_rgi70_region11_cru_1901_2019/per_glacier/RGI2000-v7.0-G-11/RGI2000-v7.0-G-11-00/RGI2000-v7.0-G-11-00001/gridded_data.nc
Exists: True


In [ ]:
# ============================================================
# 16. HISTORICAL VAS RECONSTRUCTION
# ============================================================-

print(
    f"\nRunning VAS reconstruction "
    f"{MODEL_START_YEAR}-{HISTORICAL_END_YEAR}..."
)


historical_models = (
    workflow.execute_entity_task(
        vascaling.run_historic_from_climate_data,
        gdirs,
        ys=MODEL_START_YEAR,
        ye=HISTORICAL_END_YEAR,
        climate_filename="climate_historical",
        output_filesuffix="_hist1901_2019"
    )
)



2026-10-06 11:14:49: oggm.workflow: Execute entity tasks [run_historic_from_climate_data] on 10 glaciers



Running VAS reconstruction 1901-2019...


In [ ]:
# ============================================================
# 17. CHECK HISTORICAL RESULTS
# ============================================================

successful_historical_gdirs = []

failed_historical = []


for gdir, model in zip(
    gdirs,
    historical_models
):

    if model is None:

        failed_historical.append(
            gdir.rgi_id
        )

    else:

        successful_historical_gdirs.append(
            gdir
        )


print(
    "\nHistorical reconstruction:"
)


print(
    "  Successful:",
    len(
        successful_historical_gdirs
    )
)


print(
    "  Failed:",
    len(
        failed_historical
    )
)


pd.DataFrame(
    {
        "rgi_id":
            failed_historical
    }
).to_csv(
    WORKDIR /
    "failed_historical_rgi70.csv",
    index=False
)


gdirs = (
    successful_historical_gdirs
)


if len(gdirs) == 0:

    raise RuntimeError(
        "No successful historical VAS reconstructions."
    )



In [ ]:
# ============================================================
# 18. EXTRACT HISTORICAL GLACIER RESULTS
# ============================================================

historical_glacier_results = []


for gdir in gdirs:

    diagnostic_file = (
        gdir.get_filepath(
            "model_diagnostics",
            filesuffix="_hist1901_2019"
        )
    )


    if not os.path.exists(
        diagnostic_file
    ):

        continue


    try:

        with xr.open_dataset(
            diagnostic_file
        ) as ds:

            df = pd.DataFrame(
                {
                    "year":
                        np.asarray(
                            ds["time"].values,
                            dtype=float
                        ).astype(int),

                    "volume_km3":
                        ds[
                            "volume_m3"
                        ].values
                        * 1e-9,

                    "area_km2":
                        ds[
                            "area_m2"
                        ].values
                        * 1e-6,

                    "length_km":
                        ds[
                            "length_m"
                        ].values
                        * 1e-3,

                    "specific_mb_mmwe":
                        ds[
                            "spec_mb"
                        ].values
                }
            )


        df["rgi_id"] = gdir.rgi_id

        historical_glacier_results.append(
            df
        )


    except Exception as exc:

        print(
            f"Could not read "
            f"{gdir.rgi_id}: {exc}"
        )


historical_glacier_results = (
    pd.concat(
        historical_glacier_results,
        ignore_index=True
    )
)


historical_glacier_results = (
    historical_glacier_results
    .sort_values(
        [
            "rgi_id",
            "year"
        ]
    )
    .reset_index(drop=True)
)


historical_glacier_results.to_csv(
    WORKDIR /
    "glacier_results_hist1901_2019.csv",
    index=False
)


In [ ]:
# ============================================================
# 19. AGGREGATE HISTORICAL REGIONAL RESULTS
# ============================================================

regional_df = (
    historical_glacier_results
    .groupby(
        "year",
        as_index=False
    )
    .agg(
        {
            "volume_km3":
                "sum",

            "area_km2":
                "sum",

            "length_km":
                "sum"
        }
    )
)


regional_df = (
    regional_df
    .sort_values(
        "year"
    )
    .reset_index(drop=True)
)


In [ ]:
# ============================================================
# 20. CALCULATE HISTORICAL VOLUME LOSS
# ============================================================

volume_1901 = (
    regional_df.loc[
        regional_df["year"] == MODEL_START_YEAR,
        "volume_km3"
    ]
    .iloc[0]
)


regional_df[
    "volume_loss_from_1901_km3"
] = (
    volume_1901
    -
    regional_df[
        "volume_km3"
    ]
)


regional_df[
    "mass_loss_from_1901_Gt"
] = (
    regional_df[
        "volume_loss_from_1901_km3"
    ]
    * 900.0
    * 1e-3
)



In [ ]:
# ============================================================
# 21. CALCULATE SEA-LEVEL EQUIVALENT
# ============================================================

RHO_ICE = 900.0

RHO_WATER = 1000.0

OCEAN_AREA_KM2 = 3.625e8


regional_df[
    "sea_level_change_from_1901_mm"
] = (
    regional_df[
        "volume_loss_from_1901_km3"
    ]
    * (RHO_ICE / RHO_WATER)
    / OCEAN_AREA_KM2
    * 1e6
)


regional_df.to_csv(
    WORKDIR /
    "regional_results_hist1901_2019.csv",
    index=False
)



In [ ]:
# ============================================================
# 22. PREPARE CMIP5 FILES
# ============================================================

if RUN_FUTURE:

    cmip_files = {}


    for scenario in PROJECTIONS:

        print(
            f"\nPreparing "
            f"{GCM} {scenario}..."
        )


        temp_url = (
            "https://cluster.klima.uni-bremen.de/"
            "~oggm/cmip5-ng/tas/"
            f"tas_mon_{GCM}_{scenario}_"
            "r1i1p1_g025.nc"
        )


        precip_url = (
            "https://cluster.klima.uni-bremen.de/"
            "~oggm/cmip5-ng/pr/"
            f"pr_mon_{GCM}_{scenario}_"
            "r1i1p1_g025.nc"
        )


        temp_file = (
            utils.file_downloader(
                temp_url
            )
        )


        precip_file = (
            utils.file_downloader(
                precip_url
            )
        )


        cmip_files[
            scenario
        ] = {
            "temp":
                temp_file,

            "precip":
                precip_file
        }


In [ ]:
# ============================================================
# 23. PROCESS CMIP5 CLIMATE
# ============================================================

if RUN_FUTURE:

    for scenario in PROJECTIONS:

        filesuffix = (
            f"_{GCM}_{scenario}"
        )


        print(
            f"\nProcessing CMIP5 "
            f"{GCM} {scenario}..."
        )


        workflow.execute_entity_task(
            gcm_climate.process_cmip_data,
            gdirs,
            filesuffix=filesuffix,
            fpath_temp=(
                cmip_files[
                    scenario
                ]["temp"]
            ),
            fpath_precip=(
                cmip_files[
                    scenario
                ]["precip"]
            ),
            year_range=(
                "1961",
                "1990"
            ),
            scale_stddev=True
        )



In [ ]:
# ============================================================
# 24. CHECK CMIP OUTPUTS
# ============================================================

if RUN_FUTURE:

    cmip_gdirs = {}


    for scenario in PROJECTIONS:

        filesuffix = (
            f"_{GCM}_{scenario}"
        )


        valid = []

        failed = []


        for gdir in gdirs:

            fp = (
                gdir.get_filepath(
                    "gcm_data",
                    filesuffix=filesuffix
                )
            )


            if os.path.exists(fp):

                valid.append(
                    gdir
                )

            else:

                failed.append(
                    gdir.rgi_id
                )


        print(
            f"\n{scenario}:"
        )

        print(
            "  CMIP files:",
            len(valid)
        )

        print(
            "  Missing:",
            len(failed)
        )


        pd.DataFrame(
            {
                "rgi_id":
                    failed
            }
        ).to_csv(
            WORKDIR /
            f"failed_cmip_{scenario}.csv",
            index=False
        )


        cmip_gdirs[
            scenario
        ] = valid



In [ ]:
# ============================================================
# 25. RUN FUTURE VAS PROJECTIONS
# ============================================================

if RUN_FUTURE:

    future_regional = {}


    future_glacier = {}


    for scenario in PROJECTIONS:

        print(
            f"\n========================================"
        )


        print(
            f"Running VAS {GCM} {scenario}"
        )


        print(
            f"========================================"
        )


        scenario_gdirs = (
            cmip_gdirs[
                scenario
            ]
        )


        future_suffix = (
            f"_future_{GCM}_{scenario}"
        )


        future_models = (
            workflow.execute_entity_task(
                vascaling.run_from_climate_data,
                scenario_gdirs,
                ys=FUTURE_START_YEAR,
                ye=FUTURE_END_YEAR,
                climate_filename="gcm_data",
                climate_input_filesuffix=(
                    f"_{GCM}_{scenario}"
                ),
                output_filesuffix=future_suffix,
                init_model_filesuffix="_hist1901_2019",
                init_model_yr=2019
            )
        )


        # ----------------------------------------------------
        # Read glacier results
        # ----------------------------------------------------

        glacier_rows = []


        for gdir, model in zip(
            scenario_gdirs,
            future_models
        ):

            if model is None:

                continue


            fp = (
                gdir.get_filepath(
                    "model_diagnostics",
                    filesuffix=future_suffix
                )
            )


            if not os.path.exists(fp):

                continue


            try:

                with xr.open_dataset(fp) as ds:

                    df = pd.DataFrame(
                        {
                            "year":
                                np.asarray(
                                    ds["time"].values,
                                    dtype=float
                                ).astype(int),

                            "volume_km3":
                                ds[
                                    "volume_m3"
                                ].values
                                * 1e-9,

                            "area_km2":
                                ds[
                                    "area_m2"
                                ].values
                                * 1e-6,

                            "length_km":
                                ds[
                                    "length_m"
                                ].values
                                * 1e-3
                        }
                    )


                df["rgi_id"] = (
                    gdir.rgi_id
                )


                glacier_rows.append(
                    df
                )


            except Exception as exc:

                print(
                    f"Could not read future "
                    f"output for "
                    f"{gdir.rgi_id}: {exc}"
                )


        if not glacier_rows:

            raise RuntimeError(
                f"No future results for "
                f"{scenario}."
            )


        future_glacier_results = (
            pd.concat(
                glacier_rows,
                ignore_index=True
            )
        )


        future_glacier_results = (
            future_glacier_results
            .sort_values(
                [
                    "rgi_id",
                    "year"
                ]
            )
            .reset_index(drop=True)
        )


        future_glacier[
            scenario
        ] = future_glacier_results


        # ----------------------------------------------------
        # Regional aggregate
        # ----------------------------------------------------

        future_regional_df = (
            future_glacier_results
            .groupby(
                "year",
                as_index=False
            )
            .agg(
                {
                    "volume_km3":
                        "sum",

                    "area_km2":
                        "sum",

                    "length_km":
                        "sum"
                }
            )
        )


        future_regional[
            scenario
        ] = future_regional_df


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        future_glacier_results.to_csv(
            WORKDIR /
            f"glacier_results_future_{GCM}_{scenario}.csv",
            index=False
        )


        future_regional_df.to_csv(
            WORKDIR /
            f"regional_results_future_{GCM}_{scenario}.csv",
            index=False
        )


In [ ]:
# ============================================================
# 26. CALCULATE SEA-LEVEL CHANGE FOR FUTURE SCENARIOS
# ============================================================

if RUN_FUTURE:

    for scenario in PROJECTIONS:

        df = (
            future_regional[
                scenario
            ]
            .copy()
        )


        df[
            "volume_loss_from_1901_km3"
        ] = (
            volume_1901
            -
            df["volume_km3"]
        )


        df[
            "mass_loss_from_1901_Gt"
        ] = (
            df[
                "volume_loss_from_1901_km3"
            ]
            * RHO_ICE
            * 1e-3
        )


        df[
            "sea_level_change_from_1901_mm"
        ] = (
            df[
                "volume_loss_from_1901_km3"
            ]
            * (RHO_ICE / RHO_WATER)
            / OCEAN_AREA_KM2
            * 1e6
        )


        future_regional[
            scenario
        ] = df


        df.to_csv(
            WORKDIR /
            f"regional_results_"
            f"future_sle_{GCM}_{scenario}.csv",
            index=False
        )


In [ ]:

# ============================================================
# 27. PLOT VOLUME
# ============================================================

if RUN_FUTURE:

    plt.figure(
        figsize=(11, 6)
    )


    plt.plot(
        regional_df["year"],
        regional_df["volume_km3"],
        color="goldenrod",
        linewidth=2,
        label="VAS reconstruction"
    )


    plt.plot(
        future_regional["rcp26"]["year"],
        future_regional["rcp26"]["volume_km3"],
        color="darkblue",
        linewidth=2,
        label="RCP2.6"
    )


    plt.plot(
        future_regional["rcp45"]["year"],
        future_regional["rcp45"]["volume_km3"],
        color="deepskyblue",
        linewidth=2,
        label="RCP4.5"
    )


    plt.plot(
        future_regional["rcp85"]["year"],
        future_regional["rcp85"]["volume_km3"],
        color="firebrick",
        linewidth=2,
        label="RCP8.5"
    )


    plt.axvline(
        2019,
        color="black",
        linestyle="--",
        linewidth=1
    )


    plt.xlabel("Year")

    plt.ylabel(
        "Glacier volume (km³)"
    )


    plt.title(
        f"RGI7 Region {RGI_REGION:02d} - "
        f"{REGION_NAME}\n"
        "VAS reconstruction and CMIP5 projections"
    )


    plt.xlim(
        1901,
        2100
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()

    plt.tight_layout()


    plt.savefig(
        WORKDIR /
        f"{REGION_NAME}_"
        "volume_1901_2100_RCPs.png",
        dpi=300
    )


    plt.show()


In [ ]:
# ============================================================
# 28. PLOT AREA
# ============================================================

if RUN_FUTURE:

    plt.figure(
        figsize=(11, 6)
    )


    plt.plot(
        regional_df["year"],
        regional_df["area_km2"],
        color="goldenrod",
        linewidth=2,
        label="VAS reconstruction"
    )


    plt.plot(
        future_regional["rcp26"]["year"],
        future_regional["rcp26"]["area_km2"],
        color="darkblue",
        linewidth=2,
        label="RCP2.6"
    )


    plt.plot(
        future_regional["rcp45"]["year"],
        future_regional["rcp45"]["area_km2"],
        color="deepskyblue",
        linewidth=2,
        label="RCP4.5"
    )


    plt.plot(
        future_regional["rcp85"]["year"],
        future_regional["rcp85"]["area_km2"],
        color="firebrick",
        linewidth=2,
        label="RCP8.5"
    )


    plt.axvline(
        2019,
        color="black",
        linestyle="--",
        linewidth=1
    )


    plt.xlabel("Year")

    plt.ylabel(
        "Glacier area (km²)"
    )


    plt.title(
        f"RGI7 Region {RGI_REGION:02d} - "
        f"{REGION_NAME}\n"
        "VAS reconstruction and CMIP5 projections"
    )


    plt.xlim(
        1901,
        2100
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()

    plt.tight_layout()


    plt.savefig(
        WORKDIR /
        f"{REGION_NAME}_"
        "area_1901_2100_RCPs.png",
        dpi=300
    )


    plt.show()



In [ ]:
# ============================================================
# 29. PLOT SEA-LEVEL EQUIVALENT
# ============================================================

if RUN_FUTURE:

    plt.figure(
        figsize=(11, 6)
    )


    plt.plot(
        regional_df["year"],
        regional_df[
            "sea_level_change_from_1901_mm"
        ],
        color="goldenrod",
        linewidth=2,
        label="VAS reconstruction"
    )


    for scenario, color, label in [
        (
            "rcp26",
            "darkblue",
            "RCP2.6"
        ),
        (
            "rcp45",
            "deepskyblue",
            "RCP4.5"
        ),
        (
            "rcp85",
            "firebrick",
            "RCP8.5"
        )
    ]:

        plt.plot(
            future_regional[
                scenario
            ]["year"],

            future_regional[
                scenario
            ][
                "sea_level_change_from_1901_mm"
            ],

            color=color,

            linewidth=2,

            label=label
        )


    plt.axvline(
        2019,
        color="black",
        linestyle="--",
        linewidth=1
    )


    plt.axhline(
        0,
        color="black",
        linewidth=1
    )


    plt.xlabel(
        "Year"
    )


    plt.ylabel(
        "Sea-level equivalent "
        "relative to 1901 (mm)"
    )


    plt.title(
        f"RGI7 Region {RGI_REGION:02d} - "
        f"{REGION_NAME}\n"
        "Glacier contribution to sea level"
    )


    plt.xlim(
        1901,
        2100
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()

    plt.tight_layout()


    plt.savefig(
        WORKDIR /
        f"{REGION_NAME}_"
        "sea_level_1901_2100_RCPs.png",
        dpi=300
    )


    plt.show()



In [ ]:
# ============================================================
# 30. SAVE FINAL SUMMARY
# ============================================================

if RUN_FUTURE:

    summary = []


    for scenario in PROJECTIONS:

        df = (
            future_regional[
                scenario
            ]
        )


        row2019 = df.loc[
            df["year"] == 2019
        ].iloc[0]


        row2100 = df.loc[
            df["year"] == 2100
        ].iloc[0]


        summary.append(
            {
                "RGI_region":
                    RGI_REGION,

                "region_name":
                    REGION_NAME,

                "GCM":
                    GCM,

                "scenario":
                    scenario,

                "volume_2019_km3":
                    row2019[
                        "volume_km3"
                    ],

                "volume_2100_km3":
                    row2100[
                        "volume_km3"
                    ],

                "area_2019_km2":
                    row2019[
                        "area_km2"
                    ],

                "area_2100_km2":
                    row2100[
                        "area_km2"
                    ],

                "sea_level_2100_mm":
                    row2100[
                        "sea_level_change_from_1901_mm"
                    ]
            }
        )


    summary_df = pd.DataFrame(
        summary
    )


    summary_df.to_csv(
        WORKDIR /
        f"projection_summary_{GCM}.csv",
        index=False
    )


    print(
        "\n========================================"
    )


    print(
        "FINAL PROJECTION SUMMARY"
    )


    print(
        "========================================"
    )


    print(
        summary_df.to_string(
            index=False
        )
    )


# ============================================================
# 31. FINISHED
# ============================================================

print(
    "\n========================================"
)

print(
    "RGI7 + VAS WORKFLOW COMPLETE"
)

print(
    "========================================"
)

print(
    f"\nRegion: "
    f"{RGI_REGION:02d} - {REGION_NAME}"
)

print(
    f"Historical period: "
    f"{MODEL_START_YEAR}-{HISTORICAL_END_YEAR}"
)

print(
    f"Future period: "
    f"{FUTURE_START_YEAR}-{FUTURE_END_YEAR}"
)

print(
    f"\nOutput directory:\n{WORKDIR}"
)